In [1]:
import pandas as pd
import numpy as np
import re
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
import os

# Walk up from current location until the repository folder is found
def find_repo_root():
    curr = os.path.abspath(os.getcwd())
    while True:
        # Check if we are inside the project folder
        if os.path.exists(os.path.join(curr, "data", "raw")):
            return curr
        # Check standard cloned location
        candidate = os.path.join(curr, "Documents", "ML-Lab-Final-Group-3")
        if os.path.exists(os.path.join(candidate, "data", "raw")):
            return candidate
        parent = os.path.dirname(curr)
        if parent == curr:
            break
        curr = parent
    return os.path.abspath(os.getcwd())

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
print("Active Project Directory:", os.getcwd())

Active Project Directory: C:\Users\na\Documents\ML-Lab-Final-Group-3


In [3]:

# Relative paths from the project root
RAW_PATH = os.path.join("data", "raw", "SMSSpamCollection")
PROCESSED_DIR = os.path.join("data", "processed")
REPORTS_DIR = os.path.join("reports")

# Check if the file has an accidental .txt extension added by Windows
if not os.path.exists(RAW_PATH) and os.path.exists(RAW_PATH + ".txt"):
    RAW_PATH = RAW_PATH + ".txt"

print("Loading from:", RAW_PATH)

Loading from: data\raw\SMSSpamCollection


In [4]:
import os
import pandas as pd

# 1. Ensure Python is running from the repository root (not inside notebooks/)
current_dir = os.getcwd()
if os.path.basename(current_dir) == "notebooks":
    os.chdir(os.path.abspath(os.path.join(current_dir, "..")))

print("Active Project Directory:", os.getcwd())

# 2. Define relative paths
RAW_DIR = os.path.join("data", "raw")
RAW_PATH = os.path.join(RAW_DIR, "SMSSpamCollection")
PROCESSED_DIR = os.path.join("data", "processed")
REPORTS_DIR = os.path.join("reports")

# 3. Check for .txt or detect any file inside data/raw
if not os.path.exists(RAW_PATH):
    if os.path.exists(RAW_PATH + ".txt"):
        RAW_PATH = RAW_PATH + ".txt"
    elif os.path.exists(RAW_DIR) and len(os.listdir(RAW_DIR)) > 0:
        # Pick the actual file sitting in data/raw
        RAW_PATH = os.path.join(RAW_DIR, os.listdir(RAW_DIR)[0])

print("Attempting to load from:", RAW_PATH)

# 4. Ingest dataset
df = pd.read_csv(
    RAW_PATH,
    sep="\t",
    header=None,
    names=["label", "message"],
    encoding="utf-8"
)

# 5. Ensure output folders exist
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print("Dataset loaded successfully! Shape:", df.shape)
df.head()

Active Project Directory: C:\Users\na\Documents\ML-Lab-Final-Group-3
Attempting to load from: data\raw\SMSSpamCollection
Dataset loaded successfully! Shape: (5572, 2)


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 5572
Columns: 2


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [7]:
print(df["label"].value_counts())

label
ham     4825
spam     747
Name: count, dtype: int64


In [8]:
print(df.isnull().sum())

label      0
message    0
dtype: int64


In [9]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 403


Check for empty messages

In [10]:
empty_messages = df["message"].astype(str).str.strip().eq("").sum()

print("Empty messages:", empty_messages)

Empty messages: 0


Validate the labels

In [11]:
valid_labels = {"ham", "spam"}

invalid_labels = set(df["label"].unique()) - valid_labels

print("Labels found:", df["label"].unique())
print("Invalid labels:", invalid_labels)

Labels found: ['ham' 'spam']
Invalid labels: set()


In [12]:
assert len(invalid_labels) == 0

print("Label validation passed.")

Label validation passed.


duplicate messages

In [13]:
duplicate_messages = df["message"].duplicated().sum()

print("Duplicate messages:", duplicate_messages)

Duplicate messages: 403


In [14]:
df[df["message"].duplicated(keep=False)].sort_values("message").head(10)

,label,message
505,spam,+123 Congratulations - in this week's competit...
2124,spam,+123 Congratulations - in this week's competit...
2344,ham,1) Go to write msg 2) Put on Dictionary mode 3...
1373,ham,1) Go to write msg 2) Put on Dictionary mode 3...
2163,ham,1) Go to write msg 2) Put on Dictionary mode 3...
1050,spam,18 days to Euro2004 kickoff! U will be kept in...
2719,spam,18 days to Euro2004 kickoff! U will be kept in...
2044,spam,4mths half price Orange line rental & latest c...
389,spam,4mths half price Orange line rental & latest c...
1779,ham,7 wonders in My WORLD 7th You 6th Ur style 5th...


conflicting labels

In [15]:
message_label_counts = df.groupby("message")["label"].nunique()

conflicting_messages = message_label_counts[
    message_label_counts > 1
]

print("Conflicting messages:", len(conflicting_messages))

Conflicting messages: 0


In [16]:
rows_before = len(df)

print("Rows before duplicate removal:", rows_before)

Rows before duplicate removal: 5572


In [17]:
df = df.drop_duplicates().reset_index(drop=True)

In [18]:
print("Rows after duplicate removal:", len(df))
print("Duplicates remaining:", df.duplicated().sum())

Rows after duplicate removal: 5169
Duplicates remaining: 0


clean-text column

In [19]:
def clean_text(text):
    text = str(text)
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [20]:
df["clean_message"] = df["message"].apply(clean_text)

In [21]:
df[["message", "clean_message"]].head(10)

,message,clean_message
0,"Go until jurong point, crazy.. Available only ...","go until jurong point, crazy.. available only ..."
1,Ok lar... Joking wif u oni...,ok lar... joking wif u oni...
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,U dun say so early hor... U c already then say...,u dun say so early hor... u c already then say...
4,"Nah I don't think he goes to usf, he lives aro...","nah i don't think he goes to usf, he lives aro..."
5,FreeMsg Hey there darling it's been 3 week's n...,freemsg hey there darling it's been 3 week's n...
6,Even my brother is not like to speak with me. ...,even my brother is not like to speak with me. ...
7,As per your request 'Melle Melle (Oru Minnamin...,as per your request 'melle melle (oru minnamin...
8,WINNER!! As a valued network customer you have...,winner!! as a valued network customer you have...
9,Had your mobile 11 months or more? U R entitle...,had your mobile 11 months or more? u r entitle...


In [22]:
#Checking whether cleaning created empty messages
empty_after_cleaning = (
    df["clean_message"].str.strip().eq("").sum()
)

print("Empty messages after cleaning:", empty_after_cleaning)

Empty messages after cleaning: 0


In [23]:
empty_after_cleaning = (
    df["clean_message"].str.strip().eq("").sum()
)

print("Empty messages after cleaning:", empty_after_cleaning)

Empty messages after cleaning: 0


Analyze message length

In [24]:
df["message_length"] = df["message"].str.len()

In [25]:
df["message_length"].describe()

count    5169.000000
mean       79.344554
std        58.437457
min         2.000000
25%        36.000000
50%        61.000000
75%       119.000000
max       910.000000
Name: message_length, dtype: float64

In [26]:
df.sort_values(
    "message_length",
    ascending=False
)[["label", "message", "message_length"]].head(10)

,label,message,message_length
1060,ham,For me the love should start with attraction.i...,910
1801,ham,The last thing i ever wanted to do was hurt yo...,790
2334,ham,Indians r poor but India is not a poor country...,629
1529,ham,How to Make a girl Happy? It's not at all diff...,611
2076,ham,Sad story of a Man - Last week was my b'day. M...,588
2282,ham,"Good evening Sir, hope you are having a nice d...",482
2865,ham,"&lt;#&gt; is fast approaching. So, Wish u a v...",461
1464,ham,"Hey sweet, I was wondering when you had a mome...",458
2272,ham,A Boy loved a gal. He propsd bt she didnt mind...,446
2309,ham,Solve d Case : A Man Was Found Murdered On &l...,444


Checking word count

In [27]:
df["word_count"] = df["message"].str.split().str.len()

In [28]:
df["word_count"].describe()

count    5169.000000
mean       15.439930
std        11.117073
min         1.000000
25%         7.000000
50%        12.000000
75%        22.000000
max       171.000000
Name: word_count, dtype: float64

SPLITTING THE DATA

## Converting Ham or Spam to numerical values

In [29]:
# Step A: Convert text labels to binary numbers
df["label"] = df["label"].map({"ham": 0, "spam": 1})

In [30]:
# Step B: Check the conversion
print("Data Types:\n", df.dtypes)
print("\nFirst 5 rows of labels:")
print(df[["label", "message"]].head())

print("\nValue Counts (0 = Ham, 1 = Spam):")
print(df["label"].value_counts())

Data Types:
 label              int64
message           object
clean_message     object
message_length     int64
word_count         int64
dtype: object

First 5 rows of labels:
   label                                            message
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...

Value Counts (0 = Ham, 1 = Spam):
label
0    4516
1     653
Name: count, dtype: int64


In [31]:
X = df["clean_message"]
y = df["label"]

In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [33]:
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 4135
Testing samples: 1034


In [34]:
print("Training distribution:")
print(y_train.value_counts())

print("\nTesting distribution:")
print(y_test.value_counts())

Training distribution:
label
0    3613
1     522
Name: count, dtype: int64

Testing distribution:
label
0    903
1    131
Name: count, dtype: int64


TF-IDF

In [35]:
# Vectorizer with english stopwords
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=3000
)

In [36]:
#fitting only the training data
X_train_tfidf = tfidf.fit_transform(X_train)

In [37]:
#transform the test data
X_test_tfidf = tfidf.transform(X_test)

IMPORTANT

TRAIN:
fit_transform()

TEST:
transform()

never "tfidf.fit_transform(X_test)" because that would allow the test set to influence the feature-learning process.

Check for final features

In [38]:
print("Training TF-IDF:", X_train_tfidf.shape)
print("Testing TF-IDF:", X_test_tfidf.shape)

Training TF-IDF: (4135, 3000)
Testing TF-IDF: (1034, 3000)


In [39]:
print("Vocabulary size:", len(tfidf.vocabulary_))

Vocabulary size: 3000


In [40]:
assert X_train_tfidf.shape[0] == len(y_train)
assert X_test_tfidf.shape[0] == len(y_test)
assert X_train_tfidf.shape[1] == X_test_tfidf.shape[1]

print("Feature validation passed.")

Feature validation passed.


In [41]:
#Verifiation of label feature alignment
assert X_train_tfidf.shape[0] == len(y_train)
assert X_test_tfidf.shape[0] == len(y_test)

print("Feature-label alignment verified.")

assert X_train_tfidf.shape[1] == X_test_tfidf.shape[1]

print("Train/test feature dimensions match.")

Feature-label alignment verified.
Train/test feature dimensions match.


## SAVING THE DATA

In [42]:
train_clean = pd.DataFrame({
    "message": X_train.reset_index(drop=True),
    "label": y_train.reset_index(drop=True)
})

test_clean = pd.DataFrame({
    "message": X_test.reset_index(drop=True),
    "label": y_test.reset_index(drop=True)
})

In [43]:
train_clean.to_csv(
    os.path.join(PROCESSED_DIR, "train_clean.csv"),
    index=False
)

test_clean.to_csv(
    os.path.join(PROCESSED_DIR, "test_clean.csv"),
    index=False
)

print("Clean train/test datasets saved.")

Clean train/test datasets saved.


In [44]:
import joblib
from scipy import sparse

# 1. Persist compressed sparse matrices (GitHub-safe, under 2 MB)
sparse.save_npz(os.path.join(PROCESSED_DIR, "X_train_tfidf.npz"), X_train_tfidf)
sparse.save_npz(os.path.join(PROCESSED_DIR, "X_test_tfidf.npz"), X_test_tfidf)

# 2. Persist fitted vectorizer artifact for ML Engineer (predict.py)
joblib.dump(tfidf, os.path.join(PROCESSED_DIR, "tfidf_vectorizer.joblib"))

print("Compressed sparse matrices and vectorizer artifact saved successfully!")

Compressed sparse matrices and vectorizer artifact saved successfully!


In [45]:
y_train.reset_index(drop=True).to_csv(
    "C:/Users/na/Downloads/sms+spam+collection/processed/y_train.csv",
    index=False
)

y_test.reset_index(drop=True).to_csv(
    "C:/Users/na/Downloads/sms+spam+collection/processed/y_test.csv",
    index=False
)

print("Labels saved.")

Labels saved.


## Data Quality Report

In [46]:
import pandas as pd
quality_report = pd.DataFrame({
    "Check": [
        "Initial row count",
        "Missing labels",
        "Missing messages",
        "Duplicate rows handled",
        "Conflicting duplicate labels",
        "Rows after duplicate removal",
        "Training rows",
        "Testing rows",
        "Spam class ratio in train (%)",
        "Vocabulary features"
    ],
    "Result": [
        5572,
        0,
        0,
        403,
        0,
        len(df),
        len(y_train),
        len(y_test),
        round((y_train.sum() / len(y_train)) * 100, 2),
        X_train_tfidf.shape[1]
    ],
    "Status": [
        "Checked",
        "Passed",
        "Passed",
        "Handled",
        "Passed",
        "Validated",
        "Validated",
        "Validated",
        "Stratified",
        "Optimized"
    ]
})

quality_report.to_csv(
    os.path.join(REPORTS_DIR, "data_quality_report.csv"),
    index=False
)
quality_report

Data quality report saved successfully to reports/data_quality_report.csv!


,Check,Result,Status
0,Initial row count,5572.00,Checked
1,Missing labels,0.00,Passed
2,Missing messages,0.00,Passed
3,Duplicate rows handled,403.00,Handled
4,Conflicting duplicate labels,0.00,Passed
5,Rows after duplicate removal,5169.00,Validated
6,Training rows,4135.00,Validated
7,Testing rows,1034.00,Validated
8,Spam class ratio in train (%),12.62,Stratified
9,Vocabulary features,3000.00,Optimized
